In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:100%;}
div.CodeMirror {font-family:Consolas; font-size:15pt;}
div.output {font-size:15pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:15pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render ul li{font-size:12pt;padding:5px;}
table.dataframe{font-size:15px;}
</style>
"""))

# 벡터 DB : Chroma vs Pinecone
- Chroma : 인메모리 vector DB, 로컬메모리 vector DB
- Pinecone : 클라우드 vector DB (Pinecone console에 api key 생성 -> .env (PINECONE_API_KEY등록)

## 0. 패키지 설치

In [2]:
%pip install -q pinecone-client langchain-pinecone

Note: you may need to restart the kernel to use updated packages.


## 1. Knowledge Base 구성을 위한 데이터 생성

In [3]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader('./tax_docs/with_table.docx')
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

In [4]:
len(document_list)

225

In [5]:
# embedding : upstage embedding-query
# https://python.langchain.com/v0.2/docs/integrations/text_embedding/upstage
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()
embedding = UpstageEmbeddings(
    model="solar-embedding-1-large"
    # model="embedding-query"
)

In [17]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
# 데이터를 처음 업로드할 때
index_name = "tax-index-table"
database = PineconeVectorStore.from_documents(
    documents=document_list,
    embedding=embedding,
    index_name=index_name
)
# 업로드한 벡터DB 가져올 때
# database = PineconeVectorStore(
#     embedding=embedding, # 질문을 임베딩하여 유사도 검색
#     index_name=index_name
# )

CPU times: total: 17 s
Wall time: 53.3 s


In [8]:
test_vec = embedding.embed_query("차원 확인 테스트")
print(len(test_vec))

4096


In [10]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_3NRp9v_RNS4vwStuvDqi2X7TD19NWvkjuDQE9JPB5xy8FzBb3MHEPaZXUhNr11uyQaXZ8u")
index_name = "tax-index-table"

# 새 인덱스 생성
pc.create_index(
    name=index_name,
    dimension=len(test_vec),
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
)

PineconeApiException: (409)
Reason: Conflict
HTTP response headers: HTTPHeaderDict({'content-type': 'text/plain; charset=utf-8', 'access-control-allow-origin': '*', 'vary': 'origin,access-control-request-method,access-control-request-headers', 'access-control-expose-headers': '*', 'x-pinecone-api-version': '2025-04', 'x-cloud-trace-context': 'a181d68f4e8fa1a87ff726704c27da06', 'date': 'Wed, 30 Jul 2025 07:35:01 GMT', 'server': 'Google Frontend', 'Content-Length': '85', 'Via': '1.1 google', 'Alt-Svc': 'h3=":443"; ma=2592000,h3-29=":443"; ma=2592000'})
HTTP response body: {"error":{"code":"ALREADY_EXISTS","message":"Resource  already exists"},"status":409}


In [11]:
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone(api_key="pcsk_3NRp9v_RNS4vwStuvDqi2X7TD19NWvkjuDQE9JPB5xy8FzBb3MHEPaZXUhNr11uyQaXZ8u")
index_name = "tax-index-table"

# 기존 인덱스를 연결
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

In [13]:
pc.delete_index("tax-index-table")

In [16]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key="pcsk_3NRp9v_RNS4vwStuvDqi2X7TD19NWvkjuDQE9JPB5xy8FzBb3MHEPaZXUhNr11uyQaXZ8u")

pc.create_index(
    name="tax-index-table",
    dimension=4096,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")  # 반드시 spec 지정
)

{
    "name": "tax-index-table",
    "metric": "cosine",
    "host": "tax-index-table-1rmfod6.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 4096,
    "deletion_protection": "disabled",
    "tags": null
}

## 2. 제공되는 prompt를 활용하여 답변 생성

In [18]:
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [19]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt":prompt}
)

In [20]:
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
ai_message = qa_chain.invoke({'query':query})
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '연봉 5천만원인 직장인의 소득세는 약 624만원입니다. 과세표준이 5,000만원 이상이기 때문에 5,000만원 초과 금액은 24퍼센트 세율이 적용됩니다. 따라서 세율 계산은 624만원 + (5,000만원 초과 금액의 24%)로 결정됩니다.'}